In [20]:
import pandas as pd
import numpy as np
import joblib
import glob
import hashlib
from sklearn.decomposition import TruncatedSVD
import os

# Import all modular extractors and dictionaries
from bioprocessing_eda import (
    load_and_clean_data,
    compute_esm_embeddings,
    compute_georgiev_features,
    GEORGIEV_DICT, 
    compute_aac_features,
    compute_aaindex_features,
    SelfContainedTargetTransformRegressor
)

def get_required_feature_types(features):
    required = []
    for f in features:
        f_str = str(f).upper()
        if "AAC" in f_str and 'AAC' not in required: required.append('AAC')
        if "AAINDEX" in f_str and 'AAindex' not in required: required.append('AAindex')
        if "GEORGIEV" in f_str and 'Georgiev' not in required: required.append('Georgiev')
        if "ESM" in f_str and 'ESM' not in required: required.append('ESM')
        if "SVD" in f_str and 'ESM_SVD' not in required: required.append('ESM_SVD')
    return required

def get_esm_name(features):
    if any("_ESM_Massive_3B_" in f for f in features): return "facebook/esm2_t36_3B_UR50D"
    if any("_ESM_Big_650M_" in f for f in features): return "facebook/esm2_t33_650M_UR50D"
    if any("_ESM_Large_150M_" in f for f in features): return "facebook/esm2_t30_150M_UR50D"
    if any("_ESM_Medium_35M_" in f for f in features): return "facebook/esm2_t12_35M_UR50D"
    return "facebook/esm2_t6_8M_UR50D"

def get_esm_tag(features):
    if any("_ESM_Massive_3B_" in f for f in features): return "ESM_Massive_3B"
    if any("_ESM_Big_650M_" in f for f in features): return "ESM_Big_650M"
    if any("_ESM_Large_150M_" in f for f in features): return "ESM_Large_150M"
    if any("_ESM_Medium_35M_" in f for f in features): return "ESM_Medium_35M"
    if any("_ESM_Small_8M_" in f for f in features): return "ESM_Small_8M"
    return "ESM"

def get_required_regions(features):
    regions = set()
    for f in features:
        f_str = str(f).upper()
        if 'VH' in f_str: regions.add('CD3_VH')
        if 'VL' in f_str: regions.add('CD3_VL')
        if 'SCFV' in f_str: regions.add('scFv')
    return list(regions) if regions else ['CD3_VH', 'CD3_VL', 'scFv']

def prepare_inference_data(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'CD3 VH_HCK': 'CD3_VH', 
        'CD3 VL_HCK': 'CD3_VL', 
        'CD3_VH_HCK': 'CD3_VH', 
        'CD3_VL_HCK': 'CD3_VL'
    })
    
    if 'CD3_VH' in df.columns and 'CD3_VL' in df.columns:
        def build_fv(r):
            vh = str(r['CD3_VH']).strip().upper() if pd.notna(r['CD3_VH']) else 'NAN'
            vl = str(r['CD3_VL']).strip().upper() if pd.notna(r['CD3_VL']) else 'NAN'
            if vh == 'NAN' or vl == 'NAN': return 'NAN'
            
            linker = ''
            if 'G4S Linker2_HCK' in df.columns and pd.notna(r['G4S Linker2_HCK']):
                val = str(r['G4S Linker2_HCK']).strip().upper()
                if val not in ['NAN', 'NONE']: linker = val
            return vh + linker + vl
            
        df['scFv'] = df.apply(build_fv, axis=1)
    return df

def fit_base_svd(df_base, region_col, esm_model_name, esm_tag, prefix, target_indices=None):
    print(f" ⚙️ Fitting SVD for {prefix} using {esm_model_name}...")
    valid_seqs = df_base[region_col].fillna("").astype(str)
    _, raw_matrix = compute_esm_embeddings(
        sequences=valid_seqs, esm_model_name=esm_model_name, 
        prefix=prefix, esm_tag=esm_tag, target_indices=target_indices
    )
    svd = TruncatedSVD(n_components=50, random_state=42)
    svd.fit(raw_matrix)
    return svd

def extract_features_for_model(df_test, pkg, esm_name, esm_tag, ftypes, target_regions, fitted_svds, my_target_indices=None):
    df_features = pd.DataFrame(index=df_test.index)
    
    for col in target_regions:
        valid_seqs = df_test[col].fillna("").astype(str)
        
        extraction_passes = [(False, f"{col}_", None)]
        if my_target_indices and col in my_target_indices:
            semantic_tag, target_indices = my_target_indices[col]
            short_hash = hashlib.md5(str(target_indices).encode('utf-8')).hexdigest()[:6]
            t_prefix = f"{col}_Targeted_{semantic_tag}_{short_hash}_"
            extraction_passes.append((True, t_prefix, target_indices))
            
        for is_targeted, prefix, t_idx in extraction_passes:
            if not any(f.startswith(prefix) for f in pkg['features']):
                continue
                
            if 'ESM' in ftypes:
                print(f" ⚙️ Computing ESM for {prefix}...")
                esm_dict, raw_matrix = compute_esm_embeddings(
                    sequences=valid_seqs, esm_model_name=esm_name, 
                    prefix=prefix, esm_tag=esm_tag, target_indices=t_idx
                )
                if 'ESM_SVD' in ftypes:
                    svd_transformed = fitted_svds[prefix].transform(raw_matrix)
                    for i in range(svd_transformed.shape[1]):
                        df_features[f"{prefix}{esm_tag}_SVD50_{i}"] = svd_transformed[:, i]
                else:
                    df_features = pd.concat([df_features, pd.DataFrame(esm_dict, index=df_test.index)], axis=1)
                    
            if 'Georgiev' in ftypes:
                print(f" ⚙️ Computing Georgiev for {prefix}...")
                geo_out = compute_georgiev_features(valid_seqs, GEORGIEV_DICT, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(geo_out, index=df_test.index)], axis=1)

            if 'AAC' in ftypes:
                print(f" ⚙️ Computing AAC for {prefix}...")
                aac_out = compute_aac_features(valid_seqs, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(aac_out, index=df_test.index)], axis=1)
                
            if 'AAindex' in ftypes:
                print(f" ⚙️ Computing AAindex for {prefix}...")
                try:
                    from bioprocessing_eda import aaindex_dict
                    idx_out = compute_aaindex_features(valid_seqs, aaindex_dict, prefix=prefix, target_indices=t_idx)
                except (ImportError, TypeError):
                    idx_out = compute_aaindex_features(valid_seqs, prefix=prefix, target_indices=t_idx)
                df_features = pd.concat([df_features, pd.DataFrame(idx_out, index=df_test.index)], axis=1)

    return df_features.loc[:, ~df_features.columns.duplicated()]

# 🌟 NEW: Multi-Sheet Excel Evaluation Function
def evaluate_and_save(df_test, predictions, actual_col_keyword, output_excel, target_name, percent_range):
    print(f"\n--- Evaluating {target_name} Performance ---")
    
    seq_ids = df_test['ID'] if 'ID' in df_test.columns else (df_test['Samples'] if 'Samples' in df_test.columns else df_test.index)
    base_results_df = pd.DataFrame({
        'Sequence_ID': seq_ids,
        f'Predicted_{target_name}': predictions
    })
    
    actual_col = next((c for c in df_test.columns if actual_col_keyword.lower() in c.lower()), None)
    
    if not actual_col:
        print(f"⚠️ Actual column for '{actual_col_keyword}' not found in the test dataset. Skipping evaluation.")
        return
        
    base_results_df[f'Actual_{target_name}'] = df_test[actual_col]
    base_results_df[f'Error_{target_name}'] = np.abs(base_results_df[f'Actual_{target_name}'] - base_results_df[f'Predicted_{target_name}'])
    
    base_results_df['Actual_Rank'] = base_results_df[f'Actual_{target_name}'].rank(method='min')
    base_results_df['Predicted_Rank'] = base_results_df[f'Predicted_{target_name}'].rank(method='min')
    
    # Calculate overall Spearman Correlation
    overall_spearman = base_results_df[f'Actual_{target_name}'].corr(base_results_df[f'Predicted_{target_name}'], method='spearman')
    print(f"Overall Spearman Rank Correlation on Unseen Data: {overall_spearman:.3f}")
    
    summary_data = []

    # Write multi-sheet Excel
    with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
        
        # 1. Generate individual detail sheets for each percentage
        for p in percent_range:
            top_k = max(1, int(len(base_results_df) * p))
            
            sheet_df = base_results_df.copy()
            sheet_df['Is_Hit'] = (sheet_df['Actual_Rank'] <= top_k) & (sheet_df['Predicted_Rank'] <= top_k)
            
            hits = sheet_df['Is_Hit'].sum()
            hit_percentage = (hits / top_k) * 100 if top_k > 0 else 0
            
            print(f"Hit Rate (Top {p*100:.0f}% / {top_k} sequences): {hits}/{top_k} ({hit_percentage:.1f}%)")
            
            # Append to summary metrics
            summary_data.append({
                'Top Tier Target': f"Top {int(p*100)}% (N={top_k})",
                'Hit Rate (Fraction & %%)': f"{hits}/{top_k} ({hit_percentage:.1f}%)"
            })
            
            # Save the detailed dataframe to its own sheet
            sheet_name = f"Top_{int(p*100)}_Percent"
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
        # 2. Generate the Summary Sheet
        summary_df = pd.DataFrame(summary_data)
        
        # Add a blank spacer row, then append the overall Spearman correlation
        summary_df.loc[len(summary_df)] = ["", ""]
        summary_df.loc[len(summary_df)] = ["Overall Spearman Correlation", f"{overall_spearman:.3f}"]
        
        # Ensure Summary is the first sheet by manipulating the dictionary
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        
        # Re-order worksheets so 'Summary' appears first when opened
        workbook = writer.book
        summary_sheet = workbook['Summary']
        workbook._sheets.remove(summary_sheet)
        workbook._sheets.insert(0, summary_sheet)

    print(f"🚀 Success! {target_name} comprehensive results saved to {output_excel}")

def main():
    base_csv_poly = "data/tubespin_subset.csv" 
    base_csv_hmw = "data/50-50_sequences_subset.csv" 
    
    percent_range = [0.20, 0.30, 0.40, 0.50, 0.60]
    
    my_target_indices = {
        'CD3_VH': ('Interface_looseness', [34, 36, 38, 42, 43, 44, 45, 46, 49, 60, 61, 62, 63, 96, 101, 102, 103, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117]),
        'CD3_VL': ('Interface_looseness', [30, 33, 34, 35, 36, 37, 39, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 54, 55, 56, 57, 88, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101])
    }
    
    # ==========================================
    # 1. POLYREACTIVITY PIPELINE
    # ==========================================
    print("\n🔹 Running Polyreactivity Prediction Pipeline...")
    poly_holdout_csv = "data/tubespin_unseen.csv" 
    
    # 🌟 NEW: Extract the exact model filename to use for the Excel output
    poly_model_path = glob.glob("trained_models/*ELISA_Polyreactivity_Excell*.joblib")[0]
    poly_model_basename = os.path.splitext(os.path.basename(poly_model_path))[0]
    poly_output_excel = f"{poly_model_basename}_Predictions.xlsx"
    
    pkg_poly = joblib.load(poly_model_path)
    ftypes_poly = get_required_feature_types(pkg_poly['features'])
    esm_name_poly = get_esm_name(pkg_poly['features'])
    esm_tag_poly = get_esm_tag(pkg_poly['features'])
    regions_poly = get_required_regions(pkg_poly['features'])
    
    fitted_svds_poly = {}
    if 'ESM_SVD' in ftypes_poly:
        df_base = prepare_inference_data(base_csv_poly)
        for reg in regions_poly:
            fitted_svds_poly[f"{reg}_"] = fit_base_svd(df_base, reg, esm_name_poly, esm_tag_poly, prefix=f"{reg}_")
            if reg in my_target_indices:
                sem_tag, t_idx = my_target_indices[reg]
                short_hash = hashlib.md5(str(t_idx).encode('utf-8')).hexdigest()[:6]
                t_prefix = f"{reg}_Targeted_{sem_tag}_{short_hash}_"
                fitted_svds_poly[t_prefix] = fit_base_svd(df_base, reg, esm_name_poly, esm_tag_poly, prefix=t_prefix, target_indices=t_idx)
            
    df_test_poly = prepare_inference_data(poly_holdout_csv)
    
    X_poly = extract_features_for_model(df_test_poly, pkg_poly, esm_name_poly, esm_tag_poly, ftypes_poly, regions_poly, fitted_svds_poly, my_target_indices)
    pred_poly = pkg_poly['model'].predict(X_poly[pkg_poly['features']]).flatten()
    
    # 🌟 NEW: Pass the dynamic Excel filename
    # evaluate_and_save(df_test_poly, pred_poly, 'Poly', poly_output_excel, 'Poly', percent_range)

    # ==========================================
    # 2. HMW PIPELINE
    # ==========================================
    print("\n🔹 Running HMW Prediction Pipeline...")
    hmw_holdout_csv = "data/50-50_sequences_unseen.csv" 
    
    # 🌟 NEW: Extract the exact model filename to use for the Excel output
    hmw_model_path = glob.glob("trained_models/*HMW*.joblib")[0]
    hmw_model_basename = os.path.splitext(os.path.basename(hmw_model_path))[0]
    hmw_output_excel = f"{hmw_model_basename}_Predictions.xlsx"
    
    pkg_hmw = joblib.load(hmw_model_path)
    ftypes_hmw = get_required_feature_types(pkg_hmw['features'])
    esm_name_hmw = get_esm_name(pkg_hmw['features'])
    esm_tag_hmw = get_esm_tag(pkg_hmw['features'])
    regions_hmw = get_required_regions(pkg_hmw['features'])
    
    fitted_svds_hmw = {}
    if 'ESM_SVD' in ftypes_hmw:
        df_base = prepare_inference_data(base_csv_hmw)
        for reg in regions_hmw:
            fitted_svds_hmw[f"{reg}_"] = fit_base_svd(df_base, reg, esm_name_hmw, esm_tag_hmw, prefix=f"{reg}_")
            if reg in my_target_indices:
                sem_tag, t_idx = my_target_indices[reg]
                short_hash = hashlib.md5(str(t_idx).encode('utf-8')).hexdigest()[:6]
                t_prefix = f"{reg}_Targeted_{sem_tag}_{short_hash}_"
                fitted_svds_hmw[t_prefix] = fit_base_svd(df_base, reg, esm_name_hmw, esm_tag_hmw, prefix=t_prefix, target_indices=t_idx)
            
    df_test_hmw = prepare_inference_data(hmw_holdout_csv)
    
    X_hmw = extract_features_for_model(df_test_hmw, pkg_hmw, esm_name_hmw, esm_tag_hmw, ftypes_hmw, regions_hmw, fitted_svds_hmw, my_target_indices)
    pred_hmw = pkg_hmw['model'].predict(X_hmw[pkg_hmw['features']]).flatten()
    
    # 🌟 NEW: Pass the dynamic Excel filename
    evaluate_and_save(df_test_hmw, pred_hmw, 'HMW', hmw_output_excel, 'HMW', percent_range)

if __name__ == "__main__":
    main()


🔹 Running Polyreactivity Prediction Pipeline...
 ⚙️ Computing Georgiev for CD3_VL_...
 ⚙️ Computing Georgiev for CD3_VH_...

🔹 Running HMW Prediction Pipeline...
 ⚙️ Computing ESM for scFv_...
 ⚙️ Computing AAC for scFv_...

--- Evaluating HMW Performance ---
Overall Spearman Rank Correlation on Unseen Data: 0.179
Hit Rate (Top 20% / 5 sequences): 1/5 (20.0%)
Hit Rate (Top 30% / 7 sequences): 3/7 (42.9%)
Hit Rate (Top 40% / 10 sequences): 4/10 (40.0%)
Hit Rate (Top 50% / 12 sequences): 7/12 (58.3%)
Hit Rate (Top 60% / 15 sequences): 9/15 (60.0%)
🚀 Success! HMW comprehensive results saved to Production_global_PLSRegression_SUBSET_50-50_HMW%_scFv_AAC-ESM_Small_8M_Predictions.xlsx
